# Notebook 01 — Inspección inicial

**Objetivo de esta etapa:** Comprender la estructura y calidad inicial del dataset sin tomar decisiones definitivas. El propósito es reunir evidencia para orientar las etapas posteriores.

**Preguntas de análisis definidas por el grupo:**
1. ¿Fumar es el factor que más influye en los costos médicos?
2. ¿A mayor IMC y edad, mayores costos? ¿Se potencian entre sí?
3. ¿Existen grupos diferenciados de pacientes según su perfil de riesgo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/reporte_clinica.csv')
print('Dataset cargado correctamente.')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

## 1. Estructura general

In [ ]:
print('Primeras filas:')
df.head(10)

In [ ]:
print('Tipos de datos y valores no nulos:')
df.info()

In [ ]:
print('Estadísticas descriptivas — variables numéricas:')
df.describe()

## 2. Variables y tipos de datos

In [ ]:
print('Variables identificadas:')
for col in df.columns:
    dtype = df[col].dtype
    nunique = df[col].nunique()
    print(f'  {col}: tipo={dtype}, valores únicos={nunique}')

**Observación:** El dataset contiene 7 variables analíticas y 1 columna de índice sin valor (`Unnamed: 0`). Las variables numéricas son `age`, `bmi`, `children` y `charges`. Las variables categóricas son `sex`, `smoker` y `region`. La variable objetivo para el análisis es `charges` (costo médico del paciente).

## 3. Valores faltantes

In [ ]:
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})
print('Valores faltantes por columna:')
print(resumen_nulos[resumen_nulos['Nulos'] > 0])

**Observación:** Se detectaron valores nulos en `age` (102 registros, 7.5%) y `bmi` (50 registros, 3.7%). Estas variables requerirán una decisión de imputación o eliminación en la etapa de calidad y limpieza. El resto de las variables no presenta nulos.

## 4. Duplicados

In [ ]:
n_dup = df.duplicated().sum()
print(f'Registros duplicados: {n_dup}')

**Observación:** No se detectaron filas completamente duplicadas. No se requiere acción en esta dimensión.

## 5. Valores únicos en variables categóricas

In [ ]:
categoricas = ['sex', 'smoker', 'region']
for col in categoricas:
    print(f'\n{col}: {list(df[col].unique())}')

**Observación:** Se detectaron inconsistencias de capitalización en `sex` (`female`/`Female`, `male`/`Male`), `smoker` (`yes`/`Yes`, `no`/`No`) y en `region` se encontraron abreviaciones (`SE`, `NE`, `SW`, `NW`) mezcladas con nombres completos. Estas inconsistencias generan categorías duplicadas que deben normalizarse antes del análisis.

## 6. Outliers iniciales en variables numéricas

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
numericas = ['age', 'bmi', 'children', 'charges']
for ax, col in zip(axes, numericas):
    df[col].dropna().plot(kind='box', ax=ax, title=col)
plt.suptitle('Boxplots — detección inicial de outliers', y=1.02)
plt.tight_layout()
plt.show()

print('\nValores extremos detectados:')
print(f'bmi máximo: {df["bmi"].max()}')
print(f'children mínimo: {df["children"].min()}')
print(f'children máximo: {df["children"].max()}')

**Observación:** Se detectaron valores claramente erróneos en `bmi` (valor 999.0, que es biológicamente imposible) y en `children` (valores -1 imposibles y 99 inverosímil). Estas anomalías serán tratadas en la etapa de limpieza con decisiones justificadas.

## 7. Resumen de observaciones iniciales

| Aspecto | Observación |
|---|---|
| Dimensiones | 1363 filas × 8 columnas |
| Nulos | age (102), bmi (50) |
| Duplicados | Ninguno |
| Inconsistencias categóricas | sex, smoker, region (capitalización y abreviaciones) |
| Outliers erróneos | bmi=999, children=-1, children=99 |
| Columna sin valor | Unnamed: 0 (índice redundante) |

Estas observaciones orientan la etapa de calidad y limpieza. Las decisiones específicas se documentarán con su evidencia y justificación en el notebook 02.